# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset (Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

The dataset source follows the [Croissant schema](https://mlcommons.org/croissant/) for ML-ready and FAIR dataset packaging, accessible via its schema URL.

### Dataset Source
- Croissant schema URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load Croissant metadata and create a `mlcroissant.Dataset` object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description from the metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

In Croissant, datasets can contain one or more "record sets" described in their metadata. Each record set and field is uniquely identified by its `@id`.

Let's inspect available record sets, their IDs, and fields. All references will use their exact `@id` values.

In [ ]:
from pprint import pprint

# List available record sets and their @id
record_sets = list(dataset.record_sets())
print('Record sets available in this dataset:')
for rs in record_sets:
    print(f"  - {rs['@id']}")

# For each, display their fields and columns (@id)
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs['@id']}")
    print(f"  Name: {rs.get('name', 'No name')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, str):
            print(f"    Field @id: {field}")
        elif isinstance(field, dict):
            print(f"    Field @id: {field.get('@id')}, name: {field.get('name')}")

## 3. Data Extraction

Let's extract all available records for each record set using their `@id`, and load them into pandas DataFrames keyed by record set `@id`.

For this dataset, for demonstration we'll use all record sets discovered above.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows from record set: {record_set_id}")
    else:
        print(f"Record set {record_set_id} contains no records.")

# For demonstration, show columns and first rows for each non-empty record set
for rs_id, df in dataframes.items():
    print(f"\nRecord set @id: {rs_id}")
    print(f"Columns: {list(df.columns)}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's perform common EDA steps:
- Filtering numeric records
- Normalizing a numeric field
- Grouping by a key attribute

We will pick an example record set and numeric field based on what is available above (adjust the cell if your dataset uses different names or types).

In [ ]:
# Example: Automatically select a numeric column if available
import numpy as np

if dataframes:
    rs_id = next(iter(dataframes))  # just pick the first non-empty record set
    df = dataframes[rs_id]
    # Try to pick a numeric column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col].dropna()):
            numeric_field = col
            break

    print(f"Analyzing Numeric Field @id: {numeric_field} from Record Set @id: {rs_id}\n")

    # Basic Filtering
    if numeric_field is not None:
        mean_val = df[numeric_field].mean()
        threshold = mean_val if pd.notnull(mean_val) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        # Choose a group-able field (categorical/text)
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
else:
    print("No non-empty record sets found in the dataset.")

## 5. Visualization

Let's plot the distribution of the selected numeric field (if available) and one grouped relationship.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if dataframes and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If a group_field was found, plot group means
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to use the `mlcroissant` library to discover, load, and examine the FAIR² dataset on rangeland management knowledge adoption in Northern Kenya.

- **Data Access**: Leveraged Croissant schema and `@id` referencing for robust, schema-driven data access and extraction.
- **Data Overview**: Inspected record sets, fields, and columns programmatically.
- **Exploration**: Conducted basic data cleaning, normalization, and grouping to prepare for further domain analysis.

**Next Steps:** For applied research or machine learning, enrich this notebook with domain-specific analyses or modeling using extracted DataFrames, referencing columns only by their `@id` per FAIR data practices.